In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset, Subset
from torchvision import datasets, transforms
from torchvision.transforms import v2
from functools import partial
import os
import random
import numpy as np
from typing import Literal

SEED = 0

torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(SEED)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    torch.set_deterministic_debug_mode(True)



################################
#     RESTART     RUNTIME      #
################################
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.train.train import *

import torchvision.models as models

In [2]:
DATASET_NAME: Literal['cifar10', 'cifar100'] = "cifar10"
model_type: Literal['vgg19', 'pure-ssonn', 'resnext50'] = "pure-ssonn"


In [3]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [4]:
if DATASET_NAME == "cifar100" and model_type == "vgg19":
    BATCH_SIZE = 128
else:
    BATCH_SIZE = 256

if DATASET_NAME == "cifar100":
    norm_mean = (0.5071, 0.4867, 0.4408)
    norm_std = (0.2675, 0.2565, 0.2761)
    dataset_class = datasets.CIFAR100
    classes = [str(i) for i in range(100)]
    num_classes = 100
else:
    norm_mean = (0.4914, 0.4822, 0.4465)
    norm_std = (0.2023, 0.1994, 0.2010)
    dataset_class = datasets.CIFAR10
    classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
    num_classes = 10

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std),
    v2.RandomErasing(p=0.5, scale=(0.25, 0.25), ratio=(1.0, 1.0), value=0),
])
criterion = nn.CrossEntropyLoss()

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std),
])

from torch.utils.data import random_split

full_train_dataset = dataset_class(root='./data', train=True,
                                   download=True, transform=transform_train)

test_dataset = dataset_class(root='./data', train=False,
                             download=True, transform=transform_test)

train_dataset, val_dataset = full_train_dataset, test_dataset

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
finder_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [5]:
class ExpandingHead(nn.Module):
    def __init__(self, model_type="vgg19", num_classes=10):
        super(ExpandingHead, self).__init__()
    
        if model_type == "vgg19":
            backbone = models.vgg19_bn(weights=None)
            self.backbone = backbone.features
            in_features = 512
        else:
            backbone = models.resnext50_32x4d(weights=None)
            backbone.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
            backbone.maxpool = nn.Identity()
            
            self.backbone = nn.Sequential(*list(backbone.children())[:-1])
            in_features = 2048

        self.fc0 = nn.Linear(in_features, num_classes)
    
    def forward(self, x):
        x = self.backbone(x)
        x = torch.flatten(x, 1)
        x = self.fc0(x)
        return x
    
    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

In [6]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=3 * 32 * 32, output_size=10):
        super(SimpleFCN, self).__init__()
        self.fc0 = nn.Linear(input_size, output_size)

    def forward(self, x: torch.Tensor):
        x = torch.flatten(x, start_dim=1) 
        x = self.fc0(x)
        return x


In [7]:
def init_weights(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
            
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)
        
    elif isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)


In [8]:
if model_type == "vgg19":
    model = ExpandingHead(model_type=model_type, num_classes=num_classes)
    if DATASET_NAME == "cifar10":
        hyperparams = {
            "num_epochs": 500,
            "batch_size": BATCH_SIZE,
            "edge_importance_metric": AbsGradientEdgeMetric(criterion),
            "edge_score_aggregation": "variance",
            "expansion_thresholds": {"fc0": 0.0000001},
            "pruning_thresholds": {"fc0": 1},
            "plateau_threshold": 0.01,
            # "min_epochs_between_expansions": 16,
            "min_epochs_between_expansions": 1000000,
            "plateau_window_size": 5,
            "learning_rate": 1e-2,
            "prune_after_epochs": 4,
            "task_type": "classification",
            "start_fully_connected": False,
            "max_new_edges_per_expansion": 10000,
            "weight_decay": 0,
            "diff_threshold": 0.05,
            "t_max": 100,
        }
    else:
        hyperparams = {
            "num_epochs": 700,
            "batch_size": BATCH_SIZE,
            "edge_importance_metric": AbsGradientEdgeMetric(criterion),
            "edge_score_aggregation": "variance",
            "expansion_thresholds": {"fc0": 0.00001},
            "pruning_thresholds": {"fc0": 1},
            "plateau_threshold": 0.01,
            # "min_epochs_between_expansions": 16,
            "min_epochs_between_expansions": 1000000,
            "plateau_window_size": 5,
            "learning_rate": 1.25e-2,
            "prune_after_epochs": 4,
            "task_type": "classification",
            "start_fully_connected": False,
            "max_new_edges_per_expansion": 4000,
            "weight_decay": 5e-4,
            "diff_threshold": 0.05,
            "t_max": 100,
        }
elif model_type == 'resnext50':
    model = ExpandingHead(model_type=model_type, num_classes=num_classes)
    if DATASET_NAME == "cifar10":
        hyperparams = {
            "num_epochs": 500,
            "batch_size": 256,
            "edge_importance_metric": AbsGradientEdgeMetric(criterion),
            "edge_score_aggregation": "variance",
            "expansion_thresholds": {"fc0": 1e-5},
            "pruning_thresholds": {"fc0": 1},
            "plateau_threshold": 0.01,
            "min_epochs_between_expansions": 1000000,
            "plateau_window_size": 5,
            "learning_rate": 0.1,
            "prune_after_epochs": 4,
            "task_type": "classification",
            "start_fully_connected": False,
            "max_new_edges_per_expansion": 4000,
            "weight_decay": 0.0005,
            "diff_threshold": 0.05,
            "t_max": 100,
        }
    else:
        hyperparams = {
            "num_epochs": 700,
            "batch_size": 256,
            "edge_importance_metric": AbsGradientEdgeMetric(criterion),
            "edge_score_aggregation": "variance",
            "expansion_thresholds": {"fc0": 1e-5},
            "pruning_thresholds": {"fc0": 1},
            "plateau_threshold": 0.01,
            "min_epochs_between_expansions": 1000000,
            "plateau_window_size": 5,
            "learning_rate": 0.1,
            "prune_after_epochs": 4,
            "task_type": "classification",
            "start_fully_connected": False,
            "max_new_edges_per_expansion": 4000,
            "weight_decay": 0.0005,
            "diff_threshold": 0.05,
            "t_max": 100,
        }
elif model_type == 'pure-ssonn':
    model = SimpleFCN()
    if DATASET_NAME == "cifar10":
        hyperparams = {
            "num_epochs": 700,
            "batch_size": BATCH_SIZE,
            "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
            "edge_score_aggregation": "variance",
            "expansion_thresholds": {"fc_output": 0.1},
            "pruning_thresholds": {"fc_output": 1},
            "plateau_threshold": 0.01,
            "min_epochs_between_expansions": 1000000,
            "plateau_window_size": 5,
            "learning_rate": 5e-4,
            "prune_after_epochs": 4,
            "task_type": "classification",
            "start_fully_connected": False,
            "max_new_edges_per_expansion": 2_000,
            "weight_decay": 0,
            "diff_threshold": 0.05,
            "t_max": 100,
        }

        

In [ ]:
model.apply(init_weights);
model = convert_to_scaled_net(model, layers_to_convert=[model.fc0], device=device)

In [ ]:
from comet_ml import start
from comet_ml.integration.pytorch import log_model
experiment = start(
    api_key=os.environ.get("COMET_API_KEY"),
    workspace=os.environ.get("COMET_WORKSPACE"),
    project_name="ssonn-experiments",
)
experiment.log_parameters(hyperparams)
experiment.set_name(f"{model_type}_{DATASET_NAME}_seed_{SEED}")

In [ ]:
def optimizer_factory(model_instance, current_lr=None, base_lr=None):
    if base_lr is None:
        base_lr = hyperparams['learning_rate']
    if current_lr is None:
        current_lr = base_lr

    if model_type == "vgg19":
        if DATASET_NAME == "cifar10":
            new_optimizer = optim.Adam(
                model_instance.parameters(),
                lr=current_lr,
                weight_decay=hyperparams['weight_decay'],
            )
        else:
            new_optimizer = optim.AdamW(
                model_instance.parameters(),
                lr=current_lr,
                weight_decay=hyperparams['weight_decay'],
            )
    else:
        new_optimizer = optim.SGD(
            model_instance.parameters(),
            lr=current_lr,
            weight_decay=hyperparams['weight_decay'],
            momentum=0.9
        )

    for group in new_optimizer.param_groups:
        group['initial_lr'] = base_lr 

    return new_optimizer


trained_model = train_ssonn(
    model, 
    train_loader, 
    val_loader, 
    finder_loader,
    criterion,
    optimizer_factory,
    hyperparams, 
    device,
    experiment,
    True
)

In [ ]:
experiment.end()